In [1]:
import pandas as pd

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")

In [3]:
#タイムスタンプを時系列特徴量に変換
df['time_stamp'] = pd.to_datetime(df['time_stamp'])
df['hour'] = df['time_stamp'].dt.hour
df['dayofweek'] = df['time_stamp'].dt.dayofweek

In [ ]:
# 行動に対応する重みを定義
event_weights = {
    3: 3,  # 購入
    2: 2,  # 広告クリック
    1: 1,  # 詳細ページの閲覧
    0: 0 # カートに入れる
}

In [5]:
# 重みをマッピング
df["weight"] = df["event_type"].map(event_weights)

In [6]:
# ユーザーごとに商品単位のスコアを計算
df_score = df.groupby(["user_id", "product_id"], as_index=False)["weight"].sum()
df_score.rename(columns={"weight": "score"}, inplace=True)

In [ ]:
# 順位付け（スコア降順、同スコアなら product_id 昇順）
df_score = df_score.sort_values(by=["user_id", "score", "product_id"], ascending=[True, False, True])
df_score["rank"] = df_score.groupby("user_id").cumcount() + 1

In [ ]:
#前処理結果をcsvへ
df_score.to_csv('./train/train_A.csv', index=False)